In [ ]:
import os, io, base64, numpy as np, requests, torch, torch.nn as nn
from PIL import Image

BASE_URL = os.getenv("BASE_URL", "http://ip:port") # Do not forget to change it
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

def x01_from_b64_png(b64: str) -> np.ndarray:
    """Convert base64 PNG to [0,1] numpy array.

    Args:
        b64: Base64 encoded PNG string

    Returns:
        np.ndarray: Image as (28, 28) array in [0,1] range
    """
    raw = base64.b64decode(b64)
    img = Image.open(io.BytesIO(raw)).convert("L")
    if img.size != (28, 28):
        raise ValueError("Expected 28x28 PNG")
    x = np.asarray(img, dtype=np.float32) / 255.0
    return np.clip(x, 0.0, 1.0)

def b64_png_from_x01(x2d: np.ndarray) -> str:
    """Convert [0,1] array to base64 PNG.

    Args:
        x2d: Image array in [0,1] range

    Returns:
        str: Base64 encoded PNG string
    """
    x255 = np.clip((x2d * 255.0).round(), 0, 255).astype(np.uint8)
    img = Image.fromarray(x255, mode="L")
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return base64.b64encode(buf.getvalue()).decode("ascii")

def linf(a: np.ndarray, b: np.ndarray) -> float:
    """Compute L_inf distance between two arrays.

    Args:
        a, b: Arrays to compare

    Returns:
        float: Maximum absolute difference
    """
    return float(np.max(np.abs(a - b)))

# Fetch challenge and check clean prediction
ch = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x = x01_from_b64_png(ch["image_b64"])      # (28, 28)
lab = int(ch["label"])                     # baseline label
eps = float(ch["epsilon"])                 # numeric bound
res = requests.post(f"{BASE_URL}/predict", json={"image_b64": b64_png_from_x01(x)}, timeout=10).json()
print({"baseline_label": lab, "server_pred": res["pred"], "epsilon": eps})

{'baseline_label': 1, 'server_pred': 1, 'epsilon': 0.25}


In [153]:
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x01: torch.Tensor) -> torch.Tensor:
        """Forward pass with internal normalization.

        Args:
            x01: Input tensor in [0,1] with shape (N, 1, 28, 28)

        Returns:
            Log-probabilities with shape (N, 10)
        """
        x = (x01 - MNIST_MEAN) / MNIST_STD
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

# Download and load weights
wt = requests.get(f"{BASE_URL}/weights", timeout=10).content
open("fgsm_weights.pth", "wb").write(wt)

model = SimpleClassifier().eval()
state = torch.load("fgsm_weights.pth", map_location=torch.device("cpu"))
model.load_state_dict(state)

# Verify model works locally
x_tensor = torch.from_numpy(x[None, None, ...]).float()
logits = model(x_tensor)
local_pred = int(torch.argmax(logits, dim=1).item())
print(f"Local prediction: {local_pred}, should match server: {res['pred']}")

Local prediction: 1, should match server: 1


In [154]:
# Verify server-side checks by submitting the clean image and observing expected failures.

bad = requests.post(f"{BASE_URL}/submit", json={"image_b64": b64_png_from_x01(x)}, timeout=10)
print(bad.status_code, bad.text)  # expected: 400 with "Not misclassified"

400 {"detail":"Not misclassified; try a stronger perturbation"}


In [242]:
import torch.nn.functional as F

def fgsm_attack(model, x: torch.Tensor, labels: torch.Tensor, epsilon: float, num_iter: int) -> torch.Tensor:
    """x_raw must be in [0,1] range. epsilon is applied in [0,1] pixel space."""
    x_adv = x.clone().detach().requires_grad_(True)

    MNIST_NORM_MIN = (0.0 - MNIST_MEAN) / MNIST_STD
    MNIST_NORM_MAX = (1.0 - MNIST_MEAN) / MNIST_STD
    
    alpha = (epsilon / num_iter) * 0.97 # I gust made it a little bit smaller to compensate for rounding errors
    print(alpha)
    
    for _ in range(num_iter):
        x_adv = x_adv.detach().requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, labels)
        model.zero_grad(set_to_none=True)
        loss.backward()
        x_adv = x_adv + alpha * x_adv.grad.sign()
        x_adv = torch.clamp(x + (x_adv - x).clamp(-epsilon, epsilon), MNIST_NORM_MIN, MNIST_NORM_MAX)

    print("L inf:")
    print(linf(x.detach().cpu().numpy().squeeze(), x_adv.detach().cpu().numpy().squeeze()))
    return x_adv.detach()

In [243]:
lab_tensor = torch.tensor([lab])

x_adv = fgsm_attack(model, x_tensor, lab_tensor, eps, 4)

0.060625
L inf:
0.24250006675720215


In [244]:
logits = model(x_adv)
local_pred = int(torch.argmax(logits, dim=1).item())

print(f"Local prediction: {local_pred}, should not match true label: {lab}")

Local prediction: 4, should not match true label: 1


In [245]:
x_adv_result = x_adv.detach().cpu().numpy()
x_adv_result = x_adv_result.squeeze()

b64_png_from_x01(x_adv_result)

'iVBORw0KGgoAAAANSUhEUgAAABwAAAAcCAAAAABXZoBIAAABb0lEQVR42mWRvYoVQRCFv7mOcLMzGJiewGDRhTEVgxbWSPAVfALBJ1HwSUTYxHAdUFYM1A0F0RITs1uRiMEYdPfeK3ZUP6eqzzl1RdvtbwDQ9uiIBOaftcAgmAJAQIIJWpoj6YZLTUkBgsKOgAEcpsHBATYsMGet9GEbzm7XeJ7pTzUu3PnynH214QAo5sEFQF01phIcUUvB09d1xpAbVVS2FddvnYNLZTEouSRqYmWgwKesay972Hz/9hmi69q0hkQEcY+b3SLsjQyEsCn4IY/3Qhg7blqAPzc+nGKzi4RgrFJdpd2/9qMAS/8zITFEeFmm9X3tze5GNJ8KvHvVUgFobOcCKHd/nTcvwwG5AWZRfXrEx4q9yOinUjf9zdcTA0ICW/tvC6xnJxgk6b+zPVlfdD4CVftmC7twvD5r9BK7yW/Mr7592VhIqoEBZplig3ERmg0SQx2NKShEQFn+JXIY+eAoGg4STDBnGHaQ8BePAXDgiF6yKwAAAABJRU5ErkJggg=='

In [246]:
#print(x)
#print("-----------------------------")
#print(x_adv_result)
requests.post(f"{BASE_URL}/submit", json={"image_b64": b64_png_from_x01(x_adv_result)}, timeout=10).json()

{'ok': True,
 'flag': 'HTB{f457_gr4d13n7_516n_m15l34d5_mn157}',
 'pred': 4,
 'linf': 0.24313727021217346}